# Notebook 2: Tool Use Mechanics — Dispatching, Processing, and Error Handling

**Goal**: Build the muscle memory for processing tool calls. By the end, you'll be able to:
- Write a tool dispatcher that routes calls to the right function
- Handle single and multiple tools
- Process errors gracefully with `is_error`
- Understand the full message flow

---

In [ ]:
!pip install anthropic -q
import anthropic
import json

client = anthropic.Anthropic()

## 2.1 Pattern: The Tool Dispatcher

In your interview, you'll need to connect Claude's tool requests to actual Python functions. The standard pattern is a **dispatcher** — a function that maps tool names to implementations.

There are two common approaches:

### Approach A: Dictionary mapping (cleaner)
```python
def handle_tool_call(name, input):
    handlers = {
        "get_weather": get_weather,
        "calculator": calculator,
    }
    return handlers[name](**input)
```

### Approach B: if/elif chain (simpler for few tools)
```python
def handle_tool_call(name, input):
    if name == "get_weather":
        return get_weather(**input)
    elif name == "calculator":
        return calculator(**input)
    else:
        return f"Unknown tool: {name}"
```

**During the interview, use whichever you're more comfortable with.** The dict approach scales better.

In [ ]:
# Let's build a complete example with 3 tools

# ========== TOOL IMPLEMENTATIONS ==========

def get_weather(location, unit="fahrenheit"):
    """Fake weather API."""
    # In a real app, you'd call an actual API
    weather_data = {
        "San Francisco, CA": {"temp": 62, "condition": "foggy"},
        "New York, NY": {"temp": 45, "condition": "clear"},
        "London, UK": {"temp": 50, "condition": "rainy"},
    }
    data = weather_data.get(location, {"temp": 70, "condition": "sunny"})
    if unit == "celsius":
        data["temp"] = round((data["temp"] - 32) * 5/9)
    return json.dumps({"location": location, "temperature": data["temp"], "unit": unit, "condition": data["condition"]})

def calculator(operation, a, b):
    """Basic calculator."""
    ops = {
        "add": lambda: a + b,
        "subtract": lambda: a - b,
        "multiply": lambda: a * b,
        "divide": lambda: a / b if b != 0 else "Error: division by zero",
    }
    result = ops.get(operation, lambda: f"Unknown operation: {operation}")()
    return str(result)

def get_time(timezone):
    """Fake time lookup."""
    from datetime import datetime
    # In real code you'd use pytz/zoneinfo
    return json.dumps({"timezone": timezone, "time": "2:30 PM", "date": "2025-01-15"})

print("Tool functions defined.")

In [ ]:
# ========== TOOL DEFINITIONS (for the API) ==========

tools = [
    {
        "name": "get_weather",
        "description": (
            "Get the current weather in a given location. Returns temperature, "
            "conditions, and the unit used. Use this when the user asks about weather."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state/country, e.g. 'San Francisco, CA' or 'London, UK'"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit. Defaults to fahrenheit."
                }
            },
            "required": ["location"]
        }
    },
    {
        "name": "calculator",
        "description": (
            "Performs basic arithmetic: add, subtract, multiply, divide. "
            "Use for any math calculations the user requests."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation"
                },
                "a": {"type": "number", "description": "First operand"},
                "b": {"type": "number", "description": "Second operand"}
            },
            "required": ["operation", "a", "b"]
        }
    },
    {
        "name": "get_time",
        "description": (
            "Get the current time in a given timezone. "
            "Use when the user asks what time it is somewhere."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "IANA timezone name, e.g. America/New_York"
                }
            },
            "required": ["timezone"]
        }
    }
]

print(f"Defined {len(tools)} tools: {[t['name'] for t in tools]}")

In [ ]:
# ========== THE DISPATCHER ==========

def process_tool_call(tool_name, tool_input):
    """Route a tool call to the appropriate function."""
    handlers = {
        "get_weather": get_weather,
        "calculator": calculator,
        "get_time": get_time,
    }
    
    if tool_name not in handlers:
        return f"Error: Unknown tool '{tool_name}'"
    
    try:
        # **tool_input unpacks the dict into keyword arguments
        return handlers[tool_name](**tool_input)
    except Exception as e:
        return f"Error executing {tool_name}: {str(e)}"

# Test it
print(process_tool_call("calculator", {"operation": "multiply", "a": 6, "b": 7}))
print(process_tool_call("get_weather", {"location": "San Francisco, CA"}))

## 2.2 Single Tool Call — Step by Step

Let's trace through a single tool call with careful logging so you can see every step.

In [ ]:
def single_tool_call_demo(user_message):
    """Demonstrate a single tool call with detailed logging."""
    print(f"USER: {user_message}")
    print("=" * 60)
    
    messages = [{"role": "user", "content": user_message}]
    
    # Step 1: Send to Claude
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
    
    print(f"\n[API Response]")
    print(f"  stop_reason: {response.stop_reason}")
    print(f"  content blocks: {len(response.content)}")
    
    # Step 2: Check if Claude wants to use a tool
    if response.stop_reason != "tool_use":
        print(f"  Claude responded directly: {response.content[0].text}")
        return
    
    # Step 3: Extract and log all content blocks
    for i, block in enumerate(response.content):
        if block.type == "text":
            print(f"  Block {i} [text]: {block.text}")
        elif block.type == "tool_use":
            print(f"  Block {i} [tool_use]:")
            print(f"    id:    {block.id}")
            print(f"    name:  {block.name}")
            print(f"    input: {json.dumps(block.input)}")
    
    # Step 4: Execute the tool
    tool_block = next(b for b in response.content if b.type == "tool_use")
    result = process_tool_call(tool_block.name, tool_block.input)
    print(f"\n[Tool Executed]")
    print(f"  Result: {result}")
    
    # Step 5: Build follow-up messages
    messages.append({"role": "assistant", "content": response.content})
    messages.append({
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool_block.id,
            "content": result
        }]
    })
    
    # Step 6: Get final answer
    final = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
    
    print(f"\n[Final Response]")
    print(f"  stop_reason: {final.stop_reason}")
    print(f"  text: {final.content[0].text}")
    print("=" * 60)

# Try it
single_tool_call_demo("What's the weather in San Francisco?")

In [ ]:
# Try with calculator
single_tool_call_demo("What is 17 * 23?")

## 2.3 Error Handling

Two types of errors to handle:

### Type 1: Tool execution error (your tool fails)
Return the error with `is_error: true`. Claude will apologize and explain.

### Type 2: Invalid tool call (Claude sends bad params)
Return the error message. Claude will retry with corrections (usually 2-3 times).

In [ ]:
def process_tool_call_with_errors(tool_name, tool_input):
    """Dispatcher with proper error handling."""
    handlers = {
        "get_weather": get_weather,
        "calculator": calculator,
        "get_time": get_time,
    }
    
    if tool_name not in handlers:
        # Return error tuple: (result, is_error)
        return (f"Error: Unknown tool '{tool_name}'", True)
    
    try:
        result = handlers[tool_name](**tool_input)
        return (result, False)
    except TypeError as e:
        # Missing or wrong parameters
        return (f"Error: Invalid parameters for {tool_name}: {str(e)}", True)
    except Exception as e:
        # Any other error
        return (f"Error executing {tool_name}: {str(e)}", True)

# Test error cases
print(process_tool_call_with_errors("unknown_tool", {}))
print(process_tool_call_with_errors("calculator", {"operation": "add"}))  # Missing params
print(process_tool_call_with_errors("calculator", {"operation": "divide", "a": 10, "b": 0}))

In [ ]:
# Building the tool_result message with error handling

def build_tool_result_message(tool_use_id, tool_name, tool_input):
    """Execute a tool and build the proper tool_result message."""
    result, is_error = process_tool_call_with_errors(tool_name, tool_input)
    
    tool_result = {
        "type": "tool_result",
        "tool_use_id": tool_use_id,
        "content": result,
    }
    
    if is_error:
        tool_result["is_error"] = True
    
    return tool_result

# Example
result_msg = build_tool_result_message("toolu_123", "calculator", {"operation": "add", "a": 1, "b": 2})
print(json.dumps(result_msg, indent=2))

error_msg = build_tool_result_message("toolu_456", "calculator", {"operation": "add"})
print(json.dumps(error_msg, indent=2))

## 2.4 Handling Multiple Tool_Use Blocks (Parallel Tool Calls)

Claude can return **multiple** `tool_use` blocks in a single response. This happens when it needs to call independent tools simultaneously.

### Critical Rules:
1. **All** `tool_result` blocks go in a **single** `user` message
2. `tool_result` blocks must come **first** in the content array
3. Each `tool_result` must have the matching `tool_use_id`

In [ ]:
def handle_parallel_tool_calls(response):
    """Process all tool_use blocks from a response and return a single tool_result message."""
    tool_results = []
    
    for block in response.content:
        if block.type == "tool_use":
            print(f"  Executing: {block.name}({json.dumps(block.input)})")
            result = process_tool_call(block.name, block.input)
            print(f"    -> {result}")
            
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result
            })
    
    # Return as a single user message with ALL results
    return {"role": "user", "content": tool_results}


# Test with a query that might trigger parallel tool calls
messages = [{
    "role": "user",
    "content": "What's the weather in San Francisco and what time is it there?"
}]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=tools,
    messages=messages
)

print(f"Stop reason: {response.stop_reason}")
tool_use_count = sum(1 for b in response.content if b.type == "tool_use")
print(f"Tool calls: {tool_use_count}")

if response.stop_reason == "tool_use":
    print("\nProcessing tool calls:")
    tool_result_message = handle_parallel_tool_calls(response)
    print(f"\nTool result message has {len(tool_result_message['content'])} results")
    
    # Continue conversation
    messages.append({"role": "assistant", "content": response.content})
    messages.append(tool_result_message)
    
    final = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
    print(f"\nFinal: {final.content[0].text}")

## 2.5 The Message History Pattern

This is what the full message array looks like during a tool use conversation:

```python
messages = [
    # Turn 1: User asks
    {"role": "user", "content": "What's the weather?"},
    
    # Turn 2: Claude requests tool (you add the full response.content)
    {"role": "assistant", "content": [
        {"type": "text", "text": "I'll check..."},
        {"type": "tool_use", "id": "toolu_1", "name": "get_weather", "input": {...}}
    ]},
    
    # Turn 3: You return tool result
    {"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": "toolu_1", "content": "72F sunny"}
    ]},
    
    # Turn 4: Claude gives final answer (this is the next API response)
    # {"role": "assistant", "content": [{"type": "text", "text": "It's 72F..."}]}
]
```

### Key insight: `response.content` goes directly into the assistant message
```python
messages.append({"role": "assistant", "content": response.content})
```
This is the most natural pattern — just take whatever Claude returned and append it as-is.

## 2.6 Exercise: Build a Multi-Tool Handler

Write a function `ask_claude(user_message, tools, tool_handlers)` that:
1. Sends the user message to Claude with the tools
2. If Claude wants to use a tool, executes it and returns the result
3. Returns Claude's final text response
4. Handles the case where Claude uses a tool AND where it doesn't

**This is essentially a simplified version of what you'll build in the interview.**

In [ ]:
# YOUR ANSWER
def ask_claude(user_message, tools, tool_handlers):
    """
    Send a message to Claude with tools, handle one round of tool use.
    
    Args:
        user_message: The user's question
        tools: List of tool definitions
        tool_handlers: Dict mapping tool names to functions
    
    Returns:
        Claude's final text response
    """
    pass  # YOUR CODE HERE


In [ ]:
# SOLUTION
def ask_claude(user_message, tools, tool_handlers):
    messages = [{"role": "user", "content": user_message}]
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
    
    # If Claude responds directly (no tool use)
    if response.stop_reason == "end_turn":
        return response.content[0].text
    
    # Claude wants to use tools
    if response.stop_reason == "tool_use":
        # Add assistant's response to history
        messages.append({"role": "assistant", "content": response.content})
        
        # Process ALL tool calls
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                handler = tool_handlers.get(block.name)
                if handler:
                    try:
                        result = handler(**block.input)
                    except Exception as e:
                        result = f"Error: {e}"
                else:
                    result = f"Unknown tool: {block.name}"
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        
        # Add tool results
        messages.append({"role": "user", "content": tool_results})
        
        # Get final response
        final = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        return final.content[0].text
    
    return "Unexpected response"


# Test it
handlers = {
    "get_weather": get_weather,
    "calculator": calculator,
    "get_time": get_time,
}

print(ask_claude("What is 99 * 88?", tools, handlers))
print()
print(ask_claude("What's the weather in London?", tools, handlers))

## 2.7 Extracting Tool Calls — Helper Patterns

These small utility patterns will save you time in the interview.

In [ ]:
# Pattern 1: Get all tool_use blocks from a response
def get_tool_uses(response):
    return [b for b in response.content if b.type == "tool_use"]

# Pattern 2: Get the first (or only) tool_use block
def get_first_tool_use(response):
    return next((b for b in response.content if b.type == "tool_use"), None)

# Pattern 3: Check if response has tool use
def has_tool_use(response):
    return response.stop_reason == "tool_use"

# Pattern 4: Get text from response (handles both tool_use and regular responses)
def get_response_text(response):
    text_blocks = [b.text for b in response.content if b.type == "text"]
    return "\n".join(text_blocks) if text_blocks else ""

print("Helpers defined. Use these in your interview code!")

## 2.8 Exercise: Define Tools + Handlers for a Customer Service Bot

Define 3 tools for a customer service scenario:
1. `lookup_order` — takes `order_id` (string), returns order status
2. `get_customer` — takes `customer_id` (string), returns customer info
3. `create_ticket` — takes `subject` (string), `description` (string), `priority` (enum: low/medium/high)

Write both the tool definitions AND the handler functions, then test with `ask_claude()`.

In [ ]:
# YOUR ANSWER: Define tools, handlers, and test


In [ ]:
# SOLUTION

# Handler functions
def lookup_order(order_id):
    orders = {
        "ORD-001": {"status": "shipped", "tracking": "1Z999AA10123456784", "eta": "Jan 20"},
        "ORD-002": {"status": "processing", "tracking": None, "eta": "Jan 25"},
    }
    order = orders.get(order_id, {"status": "not_found"})
    return json.dumps({"order_id": order_id, **order})

def get_customer(customer_id):
    customers = {
        "CUST-100": {"name": "Jane Doe", "email": "jane@example.com", "tier": "premium"},
    }
    customer = customers.get(customer_id, {"error": "Customer not found"})
    return json.dumps({"customer_id": customer_id, **customer})

def create_ticket(subject, description, priority="medium"):
    return json.dumps({"ticket_id": "TKT-999", "subject": subject, "priority": priority, "status": "created"})

# Tool definitions
cs_tools = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its order ID. Returns order status, tracking number, and estimated delivery date. Use when customer asks about an order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID, e.g. ORD-001"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "get_customer",
        "description": "Get customer profile by customer ID. Returns name, email, and membership tier. Use to look up customer information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID, e.g. CUST-100"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "create_ticket",
        "description": "Create a support ticket for unresolved issues. Returns the new ticket ID. Use when a customer issue cannot be resolved immediately.",
        "input_schema": {
            "type": "object",
            "properties": {
                "subject": {"type": "string", "description": "Brief subject line for the ticket"},
                "description": {"type": "string", "description": "Detailed description of the issue"},
                "priority": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Ticket priority level"
                }
            },
            "required": ["subject", "description"]
        }
    }
]

cs_handlers = {
    "lookup_order": lookup_order,
    "get_customer": get_customer,
    "create_ticket": create_ticket,
}

# Test
print(ask_claude("Can you check on my order ORD-001?", cs_tools, cs_handlers))

---

## Summary — Key Patterns for the Interview

1. **Dispatcher pattern**: Map tool names to functions with a dict
2. **`**tool_input`**: Unpack the input dict as keyword arguments
3. **Error handling**: Wrap in try/except, return `is_error: true`
4. **Parallel calls**: Collect ALL tool_use blocks, return ALL results in ONE user message
5. **Message building**: `messages.append({"role": "assistant", "content": response.content})`

**Next: Notebook 3 — The Agentic Loop (the while loop that makes everything work)**